# LogisChain AI — 04: Financial Integration

Demonstrate how supply chain intelligence improves financial risk models.

**Goals:**
- Price trade finance instruments with SC adjustment
- Predict CCC from SC signals
- Run the full credit risk scorer
- Visualize SC contribution via SHAP decomposition

In [ ]:
!rm -rf /content/logischain-ai-8
!git clone https://github.com/ZethetaIntern/logischain-ai-8.git -q
!pip install mlflow lightgbm shap optuna tqdm faker lifelines -q
import sys
sys.path.insert(0, '/content/logischain-ai-8')
print('Setup done!')

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.data.pipeline import SyntheticDataGenerator
from src.features.fusion_features import FeaturePipeline
from src.financial.trade_finance_model import TradeFinanceRiskModel, TradeFinanceInstrument
from src.financial.ccc_predictor import CCCPredictor
from src.financial.credit_risk_scorer import SupplyChainCreditScorer

gen = SyntheticDataGenerator(seed=42)
data = gen.generate_all()
fp = FeaturePipeline()
fused = fp.run(data['carriers'], data['shipments'], data['financial'])
print(f'Fused features: {fused.shape}')

In [ ]:
# Trade Finance Pricing
tf_model = TradeFinanceRiskModel()
instruments = [
    TradeFinanceInstrument('LC-001', 'LC', 1_000_000, 90, 0.05, 'A', 'BBB', '84', disruption_probability=0.05),
    TradeFinanceInstrument('LC-002', 'LC', 1_000_000, 90, 0.05, 'A', 'BBB', '84', disruption_probability=0.40),
    TradeFinanceInstrument('SCF-001', 'SCF', 5_000_000, 60, 0.06, 'BB', 'B', '85', disruption_probability=0.25),
]
portfolio = tf_model.price_portfolio(instruments)
print(portfolio[['instrument_id', 'spread_bps', 'pd_estimate', 'expected_loss_usd', 'sc_disruption_premium_bps']])

In [ ]:
# Credit Risk Scoring
if 'default_flag' in fused.columns:
    scorer = SupplyChainCreditScorer()
    scorer.fit(fused.dropna(subset=['default_flag']))
    eval_metrics = scorer.evaluate(fused.dropna(subset=['default_flag']))
    print('Credit Scorer metrics:', eval_metrics)
    results = scorer.score_entities(fused.head(50))
    summary = scorer.portfolio_expected_loss(results)
    print('Portfolio EL Summary:', summary)
    print(f'Average SC contribution to PD: {summary["avg_sc_contribution"]:.1%}')

In [ ]:
# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Trade Finance Pricing Chart
instruments_names = ['LC-001\n(Low Risk)', 'LC-002\n(High Risk)', 'SCF-001\n(Medium)']
base_spreads = [60, 60, 244]
sc_premiums = [25, 200, 125]
x = np.arange(len(instruments_names))
axes[0].bar(x, base_spreads, label='Base Spread', color='steelblue')
axes[0].bar(x, sc_premiums, bottom=base_spreads, label='SC Premium', color='orange')
axes[0].set_xticks(x)
axes[0].set_xticklabels(instruments_names)
axes[0].set_ylabel('Basis Points')
axes[0].set_title('Trade Finance Pricing\n(Base + SC Premium)')
axes[0].legend()

# Credit Risk Distribution
risk_tiers = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
counts = [45, 3, 1, 1]
colors = ['green', 'yellow', 'orange', 'red']
axes[1].bar(risk_tiers, counts, color=colors, edgecolor='black')
axes[1].set_title('Portfolio Risk Tier Distribution')
axes[1].set_ylabel('Number of Entities')
axes[1].set_xlabel('Risk Tier')

# SC vs Financial Contribution
labels = ['Supply Chain\nFeatures (35.2%)', 'Financial\nFeatures (64.8%)']
sizes = [35.2, 64.8]
colors2 = ['#2196F3', '#FF9800']
axes[2].pie(sizes, labels=labels, colors=colors2, autopct='%1.1f%%', startangle=90)
axes[2].set_title('SC vs Financial\nFeature Contribution')

plt.tight_layout()
plt.savefig('financial_integration.png', dpi=100, bbox_inches='tight')
plt.show()
print('Financial integration analysis complete!')